# Test Prediction

In [ ]:
from pathlib import Path
import os, sys, subprocess

REPO_URL = 'https://github.com/Tienkhoaa2908/ISAS2026-BLE-location-prediction-team-SQ3K.git'
REPO_NAME = 'ISAS2026-BLE-location-prediction-team-SQ3K'

if Path('/content').exists():
    work = Path('/content') / REPO_NAME
    if not work.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(work)], check=True)
    os.chdir(work)
else:
    cwd = Path.cwd()
    work = cwd if (cwd / 'requirements.txt').exists() else cwd.parent
    os.chdir(work)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('repository =', Path.cwd())


## Data preparation

In [ ]:
from pathlib import Path
import os, subprocess, sys

archive = Path(os.environ.get('SQ3K_DATA_ARCHIVE', '/content/data.zip'))
if not archive.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError('data.zip was not uploaded')
        archive = Path('/content') / next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError('Set SQ3K_DATA_ARCHIVE to the private data.zip path before running locally.')

subprocess.run([
    sys.executable, 'scripts/prepare_data.py',
    '--archive', str(archive), '--out', 'runtime/Dataset'
], check=True)
print('data_root =', Path('runtime/Dataset').resolve())


## Validation reference

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

agg = pd.read_csv('results/lodo/aggregate.csv')
by_day = pd.read_csv('results/lodo/by_day.csv')
cm = pd.read_csv('results/lodo/confusion_main_centered_normalized.csv', index_col=0)
per_class = pd.read_csv('results/lodo/per_class_main_centered.csv')

display(agg)
display(by_day)
display(per_class)

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(cm.to_numpy(), vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(cm.columns)), cm.columns, rotation=90)
ax.set_yticks(range(len(cm.index)), cm.index)
ax.set_xlabel('Predicted class')
ax.set_ylabel('True class')
ax.set_title('Fixed-22-class LODO — main + centered smoothing')
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()


## Final hidden-test inference

In [ ]:
import subprocess, sys, pandas as pd, hashlib
from pathlib import Path

out = Path('SQ3K_prediction.csv')
subprocess.run([
    sys.executable, 'scripts/test_inference.py',
    '--data-root', 'runtime/Dataset',
    '--out', str(out)
], check=True)

submission = pd.read_csv(out)
sha256 = hashlib.sha256(out.read_bytes()).hexdigest()
expected = '4b1a80a214b53b531a40467a515e2afbe4350b7b0533144ec1415c3eb9a43d80'

print('rows =', len(submission))
print('missing_predictions =', int(submission['Location'].isna().sum()))
print('predicted_classes =', submission['Location'].astype(str).nunique())
print('sha256 =', sha256)
print('matches_locked_submission =', sha256 == expected)
display(submission.head())
display(submission['Location'].astype(str).value_counts().rename('rows').to_frame())
